# 02 — Subspecialty filtering: rationale + spot-check

MedReason-Bench v1.0 evaluates frontier and open clinical LLMs on **cardiology and autoimmune** clinical reasoning. Two of our three datasets (MedQA-USMLE, PubMedQA) carry no native subspecialty labels; the third (MedMCQA) carries a `topic_name` that's free-text and inconsistent. We use a **shared keyword filter** in `data/filters/keywords.py` to mark in-scope items.

This notebook documents:
1. **The keyword lists** — what's in, what's out, and why.
2. **Recall vs. precision trade-off** — examples of items that survive the filter and items that get cut, with hand-spotted false positives / false negatives.
3. **Per-dataset diagnostic table** — for each loader, how many items the filter retains at the default `limit=100`.

Reviewers should use this notebook to flag keywords that are too narrow (false negatives) or too broad (false positives). Changes land in `data/filters/keywords.py`; this notebook gets re-run.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from data.filters.keywords import (
    AUTOIMMUNE_KEYWORDS,
    CARDIOLOGY_KEYWORDS,
    matches_subspecialty,
    specialty_for,
)
from data.loaders.medmcqa import load_medmcqa
from data.loaders.medqa import load_medqa
from data.loaders.pubmedqa import load_pubmedqa

## 1. The keyword lists

The lists deliberately favour **recall** on the first pass — better to capture a borderline item and discard it later than to miss a clinically relevant one. Keywords come from three sources:

1. **Disease names** — `myocard`, `lupus`, `vasculit`, `polymyalgia`...
2. **Anatomy / physiology fragments** — `coronary`, `aortic stenos`, `mitral`, `complement deficiency`...
3. **Diagnostic markers / drugs** — `troponin`, `bnp`, `anti-ccp`, `methotrexate`, `hydroxychloroquine`...

Excluded on purpose:
- Two-letter abbreviations (`MI`, `RA`, `CHF`) — too many false positives in non-clinical text.
- `ana` (without context) — collides with the prefix `analy*`.
- Drug classes used cross-specialty (`prednisone`, `aspirin`).

In [ ]:
print(f"Cardiology keywords ({len(CARDIOLOGY_KEYWORDS)}):")
for kw in CARDIOLOGY_KEYWORDS:
    print(f"  - {kw}")

print()
print(f"Autoimmune keywords ({len(AUTOIMMUNE_KEYWORDS)}):")
for kw in AUTOIMMUNE_KEYWORDS:
    print(f"  - {kw}")

## 2. Recall vs precision examples

In [ ]:
PROBE_TEXTS = [
    # In-scope cardio
    "A 70-year-old with substernal chest pain and ST elevation in V2-V4.",
    "Echocardiogram shows severe aortic stenosis with peak gradient 50 mmHg.",
    "Transthoracic echocardiography demonstrates dilated cardiomyopathy.",
    # In-scope autoimmune
    "Anti-CCP positive with symmetric small-joint polyarthritis.",
    "Lupus nephritis flare with rising anti-dsDNA titer.",
    "Granulomatosis with polyangiitis (Wegener's) flagged by cANCA.",
    # Out of scope
    "Acute appendicitis with McBurney's point tenderness.",
    "Streptococcus pneumoniae bacteremia treated with ceftriaxone.",
    # Borderline / known-tricky
    "Patient with chest pain after eating; suspect GERD.",  # 'chest pain' alone is not enough
    "Iron deficiency anaemia work-up in a young woman.",  # benign
]

for txt in PROBE_TEXTS:
    print(f"  match={matches_subspecialty(txt)!s:<5}  spec={specialty_for(txt)!s:<11}  | {txt}")

## 3. Per-dataset retention

What fraction of each dataset survives the filter, at the default `limit=100`?

In [ ]:
for name, loader in [
    ("medmcqa", load_medmcqa),
    ("medqa", load_medqa),
    ("pubmedqa", load_pubmedqa),
]:
    in_scope = list(loader(filter_subspecialty=True, limit=100))
    print(f"{name:>10s}: kept {len(in_scope)} (default cap 100, post-filter)")

## 4. Action items

When reviewing this notebook, log:
- **False negatives** (clinically relevant items the filter missed) → add the missing keyword to `data/filters/keywords.py`.
- **False positives** (out-of-scope items the filter kept) → either tighten the offending keyword or add a per-dataset overrides list.
- **Edge cases** (items that arguably belong to both specialties or neither) → record the call in `data/README.md`.